In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("RideAnalyticsPipeline").getOrCreate()

In [0]:
base_path = "/Volumes/dbw_de_training/default/utkarshk_analytics/"
bronze_path = base_path + "bronze/"
silver_path = base_path + "silver/"
gold_path   = base_path + "gold/"
checkpoint_path = base_path + "checkpoints/"

In [0]:
trip_schema = StructType([
    StructField("trip_id", StringType()),
    StructField("driver_id", StringType()),
    StructField("pickup_zone_id", StringType()),
    StructField("dropoff_zone_id", StringType()),
    StructField("trip_distance", DoubleType()),
    StructField("fare_amount", DoubleType()),
    StructField("trip_timestamp", TimestampType())
])

driver_schema = StructType([
    StructField("driver_id", StringType()),
    StructField("driver_name", StringType()),
    StructField("driver_rating", DoubleType())
])

zone_schema = StructType([
    StructField("zone_id", StringType()),
    StructField("zone_name", StringType()),
    StructField("city", StringType())
])

In [0]:
trips_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .schema(trip_schema)
    .load("/Volumes/dbw_de_training/default/utkarshk_analytics/")
)

In [0]:
drivers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .schema(driver_schema)
    .load("/Volumes/dbw_de_training/default/utkarshk_analytics/")
)

In [0]:
zones_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .schema(zone_schema)
    .load("/Volumes/dbw_de_training/default/utkarshk_analytics/")
)

In [0]:
trips_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path + "trips_bronze") \
    .trigger(once=True)\
    .start(bronze_path + "trips")

drivers_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path + "drivers_bronze") \
    .trigger(once=True)\
    .start(bronze_path + "drivers")

zones_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path + "zones_bronze") \
    .trigger(once=True)\
    .start(bronze_path + "zones")

In [0]:
trips_df = spark.read.format("delta").load(bronze_path + "trips")
drivers_df = spark.read.format("delta").load(bronze_path + "drivers")
zones_df = spark.read.format("delta").load(bronze_path + "zones")

In [0]:
clean_trips = (
    trips_df
    .filter("fare_amount > 0 AND trip_distance > 0")
    .dropna()
    .dropDuplicates(["trip_id"])
)

In [0]:
rejected_trips = trips_df.filter("fare_amount <= 0 OR trip_distance <= 0")

In [0]:
enriched_df = (
    clean_trips.alias("t")
    .join(drivers_df.alias("d"), "driver_id", "left")
    .join(zones_df.alias("pz"), col("t.pickup_zone_id") == col("pz.zone_id"), "left")
    .join(zones_df.alias("dz"), col("t.dropoff_zone_id") == col("dz.zone_id"), "left")
    .select(
        col("t.trip_id"),
        col("t.driver_id"),
        col("d.driver_name"),
        col("d.driver_rating"),
        col("t.trip_distance"),
        col("t.fare_amount"),
        col("t.trip_timestamp"),
        col("pz.zone_name").alias("pickup_zone"),
        col("pz.city").alias("pickup_city"),
        col("dz.zone_name").alias("dropoff_zone"),
        col("dz.city").alias("dropoff_city")
    )
)

In [0]:
enriched_df.write \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path + "silver") \
    .save(silver_path + "trips_enriched")

In [0]:
silver_df = spark.readStream.format("delta").load(silver_path + "trips_enriched")

In [0]:
driver_perf = (
    silver_df
    .groupBy("driver_id", "driver_name")
    .agg(
        count("*").alias("total_trips"),
        sum("fare_amount").alias("total_earnings"),
        avg("driver_rating").alias("avg_rating")
    )
)

In [0]:
driver_perf.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", checkpoint_path + "gold_driver") \
    .trigger(once=True)\
    .start(gold_path + "driver_performance")

In [0]:
pickup_zone = (
    silver_df
    .groupBy("pickup_zone")
    .agg(count("*").alias("total_pickups"))
)

In [0]:
drop_zone = (
    silver_df
    .groupBy("dropoff_zone")
    .agg(count("*").alias("total_dropoffs"))
)

In [0]:
revenue_city = (
    silver_df
    .groupBy("pickup_city")
    .agg(
        sum("fare_amount").alias("total_revenue"),
        avg("fare_amount").alias("avg_fare")
    )
)

In [0]:
pickup_zone.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", checkpoint_path + "pickup_zone") \
    .trigger(once=True)\
    .start(gold_path + "pickup_zone")

drop_zone.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", checkpoint_path + "drop_zone") \
    .trigger(once=True)\
    .start(gold_path + "drop_zone")

revenue_city.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", checkpoint_path + "revenue") \
    .trigger(once=True)\
    .start(gold_path + "revenue")